# Experimento — Firma digital de PDFs (pyHanko)

## Objetivo

Firmar y verificar documentos PDF reales usando certificados digitales, demostrando que una firma digital garantiza tanto la **autenticidad** (quién firmó) como la **integridad** (que el documento no fue alterado después).

## Qué hacer

Usar `pyHanko` para:
- Firmar un PDF con un certificado generado con `step-ca`
- Verificar la firma

### Contexto

Casa Monarca emite documentos críticos: constancias de atención, registros de visitas y cartas de apoyo para personas migrantes. Quien los recibe — una autoridad migratoria, un consulado, un juzgado — necesita saber que el documento es auténtico y no fue modificado.

La pregunta concreta: ¿cómo garantizar que un PDF no fue alterado después de que un coordinador autorizado lo firmó?

## Instalación

```bash
pip install pyhanko pyhanko-certvalidator cryptography reportlab fpdf2
```

---
### Paso 1 — Generación del certificado

En producción, el certificado lo emite la CA interna de Casa Monarca (step-ca). Para este experimento lo generamos directamente en Python con `cryptography`, que es la misma librería que usa pyHanko internamente.

In [ ]:
import datetime
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa

# Generamos la llave privada RSA de 2048 bits
llave_privada = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)

# Datos del firmante — simulamos una coordinadora de Casa Monarca
nombre = x509.Name([
    x509.NameAttribute(NameOID.COUNTRY_NAME, "MX"),
    x509.NameAttribute(NameOID.ORGANIZATION_NAME, "Casa Monarca"),
    x509.NameAttribute(NameOID.COMMON_NAME, "Coordinadora Legal"),
])

# Construimos el certificado X.509 autofirmado
certificado = (
    x509.CertificateBuilder()
    .subject_name(nombre)
    .issuer_name(nombre)  # autofirmado: el emisor y el sujeto son el mismo
    .public_key(llave_privada.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.utcnow())
    .not_valid_after(datetime.datetime.utcnow() + datetime.timedelta(days=365))
    .add_extension(
        x509.BasicConstraints(ca=False, path_length=None),
        critical=True,
    )
    .sign(llave_privada, hashes.SHA256())
)

# Guardamos llave y certificado en formato PEM
with open("llave.pem", "wb") as f:
    f.write(llave_privada.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.TraditionalOpenSSL,
        encryption_algorithm=serialization.NoEncryption(),
    ))

with open("certificado.pem", "wb") as f:
    f.write(certificado.public_bytes(serialization.Encoding.PEM))

print("Certificado generado para:", certificado.subject.get_attributes_for_oid(NameOID.COMMON_NAME)[0].value)
print("Válido hasta:", certificado.not_valid_after_utc)

Certificado generado para: Coordinadora Legal
Válido hasta: 2027-03-27 03:58:31+00:00


---
### Paso 2 — Generación del PDF a firmar

Creamos una constancia de atención de Casa Monarca. En el sistema real este PDF vendría del módulo de gestión de documentos.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

def generar_constancia(ruta_salida: str, nombre_migrante: str, folio: str):
    """
    Genera un PDF de constancia de atención de Casa Monarca.

    Args:
        ruta_salida (str): ruta donde se guardará el PDF
        nombre_migrante (str): nombre de la persona atendida
        folio (str): número de folio del documento
    """
    c = canvas.Canvas(ruta_salida, pagesize=letter, compress=0)
    ancho, alto = letter

    c.setFont("Helvetica-Bold", 16)
    c.drawCentredString(ancho / 2, alto - 80, "Casa Monarca")
    c.setFont("Helvetica", 12)
    c.drawCentredString(ancho / 2, alto - 100, "Ayuda Humanitaria al Migrante, A.C.")

    c.setFont("Helvetica-Bold", 13)
    c.drawCentredString(ancho / 2, alto - 140, "CONSTANCIA DE ATENCIÓN")

    c.setFont("Helvetica", 11)
    c.drawString(72, alto - 200, f"Folio: {folio}")
    c.drawString(72, alto - 230, "Por medio del presente se hace constar que:")
    c.setFont("Helvetica-Bold", 11)
    c.drawString(72, alto - 255, f"    {nombre_migrante}")
    c.setFont("Helvetica", 11)
    c.drawString(72, alto - 280, "recibió atención humanitaria, legal y social por parte de Casa Monarca.")
    c.drawString(72, alto - 320, f"Fecha de emisión: {datetime.date.today().strftime('%d de %B de %Y')}")

    # Área reservada para la firma digital
    c.setFont("Helvetica", 9)
    c.drawString(72, 120, "Firma digital del coordinador autorizado:")
    c.rect(72, 72, 300, 40)

    c.save()
    print(f"PDF generado: {ruta_salida}")


generar_constancia(
    ruta_salida="constancia.pdf",
    nombre_migrante="Juan Carlos Pérez Ramírez",
    folio="CM-2024-00847"
)

PDF generado: constancia.pdf


---
### Paso 3 — Firma del PDF con pyHanko

La firma no es una imagen ni un sello visual. Es una estructura criptográfica embebida en el PDF que vincula el contenido del documento con la identidad del firmante.

In [3]:
import asyncio
from pyhanko.sign import signers
from pyhanko.sign.fields import SigFieldSpec, append_signature_field
from pyhanko.pdf_utils.incremental_writer import IncrementalPdfFileWriter

async def firmar_pdf(ruta_entrada: str, ruta_salida: str):
    """
    Firma digitalmente un PDF usando pyHanko.

    Args:
        ruta_entrada (str): PDF sin firmar
        ruta_salida (str): PDF firmado resultante
    """
    firmante = signers.SimpleSigner.load(
        key_file="llave.pem",
        cert_file="certificado.pem",
    )

    with open(ruta_entrada, "rb") as f:
        writer = IncrementalPdfFileWriter(f)

        # Posicionamos el campo de firma en el recuadro reservado en el PDF
        append_signature_field(
            writer,
            sig_field_spec=SigFieldSpec(
                sig_field_name="FirmaCoordinador",
                box=(72, 72, 372, 112)
            )
        )

        metadatos = signers.PdfSignatureMetadata(
            field_name="FirmaCoordinador",
            reason="Emisión oficial de constancia de atención",
            location="Monterrey, Nuevo León, México",
        )

        with open(ruta_salida, "wb") as salida:
            await signers.async_sign_pdf(
                writer,
                metadatos,
                signer=firmante,
                output=salida,
            )

    print(f"PDF firmado guardado en: {ruta_salida}")


await firmar_pdf("constancia.pdf", "constancia_firmada.pdf")

PDF firmado guardado en: constancia_firmada.pdf


---
### Paso 2b — Alternativa: usar un PDF existente (para uso real)

Cuando el experimento pase a producción, el PDF no se genera desde cero sino que ya existe.
Esta celda toma ese PDF real y lo deja listo para ser firmado por el Paso 3.

In [ ]:
from fpdf import FPDF
import os

def preparar_pdf_existente(ruta_entrada: str, ruta_salida: str):
    """
    Prepara un PDF existente para ser firmado con pyHanko.
    Copia el archivo original a ruta_salida sin modificarlo,
    listo para que el Paso 3 le agregue la firma digital.

    Args:
        ruta_entrada (str): ruta al PDF real que se quiere firmar
        ruta_salida (str): debe ser constancia.pdf para que el Paso 3 lo tome
    """
    if not os.path.exists(ruta_entrada):
        print(f"No se encontró el archivo: {ruta_entrada}")
        print("Cambia ruta_entrada por la ruta completa a tu PDF.")
        print("Ejemplo: r\"C:\\Users\\Emiliano\\Documents\\mi_documento.pdf\"")
        return

    import shutil
    shutil.copy(ruta_entrada, ruta_salida)
    print(f"PDF listo para firmar: {ruta_salida}")
    print(f"Archivo original: {ruta_entrada}")


# Cambia esta ruta por la de tu PDF — usa r"..." para rutas de Windows
preparar_pdf_existente(
    ruta_entrada=r"mi_documento.pdf",
    ruta_salida="constancia.pdf"
)

---
### Paso 4 — Verificación de la firma

**Caso de prueba A — sin trust_root (resultado esperado:  Firma no válida)**

pyHanko no conoce nuestro certificado autofirmado, así que no puede confiar en él. Este resultado es correcto y esperado para el experimento — demuestra que la validación es estricta y no acepta cualquier certificado.

In [4]:
from pyhanko.pdf_utils.reader import PdfFileReader
from pyhanko.sign.validation import async_validate_pdf_signature
from pyhanko_certvalidator import ValidationContext

async def verificar_firmas(ruta_pdf: str):
    with open(ruta_pdf, "rb") as f:
        lector = PdfFileReader(f)
        firmas = lector.embedded_signatures

        if not firmas:
            print("El documento no contiene firmas digitales")
            return

        print(f"Firmas encontradas: {len(firmas)}\n")

        for i, firma in enumerate(firmas):
            contexto = ValidationContext(allow_fetching=False)
            estado = await async_validate_pdf_signature(firma, contexto)

            print(f"--- Firma #{i + 1} ---")
            print(f"  Campo:       {firma.field_name}")
            print(f"  Fecha:       {estado.signer_reported_dt}")
            print(f"  Integridad:  {'Intacto' if estado.intact else 'Modificado tras la firma'}")
            print(f"  Certificado: {'Válido' if estado.valid else 'No confiable'}")
            print(f"  Resultado:   {'Firma válida' if estado.bottom_line else 'Firma no válida'}")
            print()


await verificar_firmas("constancia_firmada.pdf")

Validation error [cert context: Common Name: Coordinadora Legal, Organization: Casa Monarca, Country: MX]: The X.509 certificate provided is self-signed - "Common Name: Coordinadora Legal, Organization: Casa Monarca, Country: MX"
Traceback (most recent call last):
  File "c:\Users\Emiliano\AppData\Local\Programs\Python\Python310\lib\site-packages\pyhanko_certvalidator\__init__.py", line 31, in find_valid_path
    async for candidate_path in paths:
  File "c:\Users\Emiliano\AppData\Local\Programs\Python\Python310\lib\site-packages\pyhanko_certvalidator\registry.py", line 689, in __anext__
    raise PathBuildingError(
pyhanko_certvalidator.errors.PathBuildingError: Unable to build a validation path for the certificate "Common Name: Coordinadora Legal, Organization: Casa Monarca, Country: MX" - no issuer matching "Common Name: Coordinadora Legal, Organization: Casa Monarca, Country: MX" was found

During handling of the above exception, another exception occurred:

Traceback (most recent 

Firmas encontradas: 1

--- Firma #1 ---
  Campo:       FirmaCoordinador
  Fecha:       2026-03-27 03:58:40+00:00
  Integridad:  Intacto
  Certificado: Válido
  Resultado:   Firma no válida



---
### Paso 4b — Verificación con trust_root (resultado esperado: Firma válida)

**Caso de prueba B — con trust_root (simulando un certificado de CA reconocida)**

Le indicamos explícitamente a pyHanko que confíe en nuestro certificado autofirmado. Esto simula lo que ocurriría en producción con un certificado emitido por la CA interna de Casa Monarca (step-ca), donde el validador conocería y confiaría en la CA raíz.

In [7]:
from pyhanko.pdf_utils.reader import PdfFileReader
from pyhanko.sign.validation import async_validate_pdf_signature
from pyhanko_certvalidator import ValidationContext
from asn1crypto import pem, x509

async def verificar_firmas_con_trust(ruta_pdf: str):
    # Parseamos el certificado PEM a objeto asn1crypto
    with open("certificado.pem", "rb") as f:
        cert_data = f.read()

    # Convertir de PEM a objeto x509 que entiende ValidationContext
    _, _, der_bytes = pem.unarmor(cert_data)
    cert_obj = x509.Certificate.load(der_bytes)

    contexto = ValidationContext(
        trust_roots=[cert_obj],
        allow_fetching=False,
    )

    with open(ruta_pdf, "rb") as f:
        lector = PdfFileReader(f)
        firmas = lector.embedded_signatures

        if not firmas:
            print("El documento no contiene firmas digitales")
            return

        print(f"🔍 Firmas encontradas: {len(firmas)}\n")

        for i, firma in enumerate(firmas):
            estado = await async_validate_pdf_signature(firma, contexto)

            print(f"--- Firma #{i + 1} ---")
            print(f"  Campo:       {firma.field_name}")
            print(f"  Fecha:       {estado.signer_reported_dt}")
            print(f"  Integridad:  {'Intacto' if estado.intact else 'Modificado tras la firma'}")
            print(f"  Certificado: {'Válido' if estado.valid else 'No confiable'}")
            print(f"  Resultado:   {'Firma válida' if estado.bottom_line else 'Firma no válida'}")
            print()


await verificar_firmas_con_trust("constancia_firmada.pdf")

🔍 Firmas encontradas: 1

--- Firma #1 ---
  Campo:       FirmaCoordinador
  Fecha:       2026-03-27 03:58:40+00:00
  Integridad:  Intacto
  Certificado: Válido
  Resultado:   Firma válida



---
### Paso 5 — ¿Qué pasa si modificamos el PDF después de firmarlo?

Simulamos que alguien altera el nombre del migrante directamente en los bytes del archivo para comprobar que pyHanko detecta el cambio.

In [8]:
import shutil

shutil.copy("constancia_firmada.pdf", "constancia_manipulada.pdf")

with open("constancia_manipulada.pdf", "r+b") as f:
    contenido = bytearray(f.read())

    objetivo = b"Juan Carlos"
    pos = contenido.find(objetivo)

    if pos != -1:
        contenido[pos:pos + len(objetivo)] = b"Pedro FALSO"
        f.seek(0)
        f.write(contenido)
        print("Modificación realizada en el PDF\n")
    else:
        print("Texto no encontrado en los bytes (puede estar comprimido)\n")

print("Resultado de verificación sobre el PDF manipulado:")
await verificar_firmas("constancia_manipulada.pdf")

Validation error [cert context: Common Name: Coordinadora Legal, Organization: Casa Monarca, Country: MX]: The X.509 certificate provided is self-signed - "Common Name: Coordinadora Legal, Organization: Casa Monarca, Country: MX"
Traceback (most recent call last):
  File "c:\Users\Emiliano\AppData\Local\Programs\Python\Python310\lib\site-packages\pyhanko_certvalidator\__init__.py", line 31, in find_valid_path
    async for candidate_path in paths:
  File "c:\Users\Emiliano\AppData\Local\Programs\Python\Python310\lib\site-packages\pyhanko_certvalidator\registry.py", line 689, in __anext__
    raise PathBuildingError(
pyhanko_certvalidator.errors.PathBuildingError: Unable to build a validation path for the certificate "Common Name: Coordinadora Legal, Organization: Casa Monarca, Country: MX" - no issuer matching "Common Name: Coordinadora Legal, Organization: Casa Monarca, Country: MX" was found

During handling of the above exception, another exception occurred:

Traceback (most recent 

Texto no encontrado en los bytes (puede estar comprimido)

Resultado de verificación sobre el PDF manipulado:
Firmas encontradas: 1

--- Firma #1 ---
  Campo:       FirmaCoordinador
  Fecha:       2026-03-27 03:58:40+00:00
  Integridad:  Intacto
  Certificado: Válido
  Resultado:   Firma no válida



---
## Criterio de éxito

El experimento es exitoso si:
- `constancia_firmada.pdf` muestra **Firma válida** e **Intacto**
- `constancia_manipulada.pdf` muestra **Modificado tras la firma**, con firma inválida

En Adobe Acrobat Reader y otros lectores compatibles con firmas digitales, el primer archivo mostrará un sello verde de firma válida y el segundo mostrará una advertencia de integridad comprometida.

## Qué aprendemos

**Firma digital ≠ imagen de firma**

Una firma escaneada o dibujada no ofrece ninguna garantía técnica — cualquiera puede copiarla y pegarla en otro documento. La firma digital de pyHanko es una operación criptográfica que une matemáticamente el contenido del PDF con la llave privada del firmante. Si el contenido cambia aunque sea un solo byte, la firma deja de ser válida.

**Integridad + autenticidad en un solo mecanismo**
- *Integridad*: el documento no fue modificado después de firmarse
- *Autenticidad*: quien firmó era quien dice ser, verificable mediante el certificado

Ambas propiedades son necesarias para que una constancia de Casa Monarca tenga valor ante terceros.

**Próximos pasos en el sistema real**

En la implementación completa, los certificados vendrán de la CA interna (step-ca) en lugar de ser autofirmados, el flujo de firma estará integrado en el backend de FastAPI, y el control de acceso (quién puede firmar qué documentos) lo gestionará Keycloak por roles.